In [2]:
# Imports
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")

All libraries imported successfully.
PyTorch version : 2.11.0+cpu


In [3]:
# Load Dataset 
train = pd.read_csv('../data/sent_train.csv')
valid = pd.read_csv('../data/sent_valid.csv')

print(f"Train shape : {train.shape}")
print(f"Valid shape : {valid.shape}")

Train shape : (9543, 2)
Valid shape : (2388, 2)


In [4]:
# Text Cleaning Function
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))
# Remove financially meaningful words from stopwords
keep_words = {'up', 'down', 'not', 'no', 'nor', 'against', 'below', 'above', 'under', 'over'}
stop_words = stop_words - keep_words

def clean_text(text):
    # Remove
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)  #URL
    text = re.sub(r'@\w+', '', text)  #@mentions
    text = re.sub(r'\$([A-Za-z]+)', r'\1', text)   #$
    text = re.sub(r'#(\w+)', r'\1', text)  #hashtags
    text = re.sub(r'[^a-z\s]', '', text)   #spl char, punc
    text = re.sub(r'\s+', ' ', text).strip()   #whitespace
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]   #topwords and short tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Test on sample tweets
samples = [
    "$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/abc123",
    "$TSLA - Tesla beats Q4 estimates, raises guidance @elonmusk #Tesla",
    "Central banks don't have as much monetary policy power as they used to."
]

print("CLEANING TEST:")
print("-" * 60)
for s in samples:
    print(f"Original : {s}")
    print(f"Cleaned  : {clean_text(s)}")
    print()

CLEANING TEST:
------------------------------------------------------------
Original : $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/abc123
Cleaned  : bynd jpmorgan reel expectation beyond meat

Original : $TSLA - Tesla beats Q4 estimates, raises guidance @elonmusk #Tesla
Cleaned  : tsla tesla beat estimate raise guidance tesla

Original : Central banks don't have as much monetary policy power as they used to.
Cleaned  : central bank dont much monetary policy power used



In [5]:
# Apply Cleaning
train['cleaned_text'] = train['text'].apply(clean_text)
valid['cleaned_text'] = valid['text'].apply(clean_text)

# Verify no empty strings after cleaning
train_empty = (train['cleaned_text'].str.strip() == '').sum()
valid_empty  = (valid['cleaned_text'].str.strip() == '').sum()

print(f"Empty texts after cleaning — Train : {train_empty}")
print(f"Empty texts after cleaning — Valid : {valid_empty}")
print()
print("Sample cleaned texts (Train):")
print("-" * 60)
for i in range(5):
    print(f"Original : {train['text'].iloc[i]}")
    print(f"Cleaned  : {train['cleaned_text'].iloc[i]}")
    print(f"Label    : {train['label'].iloc[i]}")
    print()

Empty texts after cleaning — Train : 8
Empty texts after cleaning — Valid : 0

Sample cleaned texts (Train):
------------------------------------------------------------
Original : $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Cleaned  : bynd jpmorgan reel expectation beyond meat
Label    : 0

Original : $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3
Cleaned  : ccl rcl nomura point booking weakness carnival royal caribbean
Label    : 0

Original : $CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb
Cleaned  : cemex cut credit suisse morgan weak building outlook
Label    : 0

Original : $ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N
Cleaned  : es btig research cut neutral
Label    : 0

Original : $FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB
Cleaned  : fnko funko slide piper jaffray cut
Label    : 0



In [6]:
# Handle Empty Texts
print("Empty texts in Train:")
print(train[train['cleaned_text'].str.strip() == ''][['text', 'label']])
print()

# Drop empty cleaned texts from train
train = train[train['cleaned_text'].str.strip() != ''].reset_index(drop=True)

print(f"Train shape after dropping empty texts : {train.shape}")
print(f"Valid shape unchanged                  : {valid.shape}")
print()

# Verify class distribution is not affected significantly
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
print("Train class distribution after cleaning:")
for label_id, label_name in label_map.items():
    count = (train['label'] == label_id).sum()
    pct   = count / len(train) * 100
    print(f"  {label_name:<10} : {count:>5}  ({pct:.1f}%)")

Empty texts in Train:
                                                   text  label
3943                                                 :)      2
3948                                            @TicToc      2
3949  @tictoc @telefenoticias @teleSUR_Chile @Paolad...      2
3950  @tictoc @telefenoticias @teleSUR_Chile @Paolad...      2
4440                                                 F5      2
4681                            https://t.co/575AH1YRkF      2
4682                            https://t.co/9eZPvQhfMq      2
4683                            https://t.co/oJxNPEUpWq      2

Train shape after dropping empty texts : (9535, 3)
Valid shape unchanged                  : (2388, 3)

Train class distribution after cleaning:
  Bearish    :  1442  (15.1%)
  Bullish    :  1923  (20.2%)
  Neutral    :  6170  (64.7%)


In [7]:
# Build Vocabulary

PAD_TOKEN   = '<PAD>'   # padding token
UNK_TOKEN   = '<UNK>'   # unknown token for words not in vocabulary
MIN_FREQ    = 2         # ignore words appearing less than 2 times

# Count word frequencies in train
word_freq = Counter()
for text in train['cleaned_text']:
    word_freq.update(text.split())

# Filter by minimum frequency
vocab_words = [w for w, freq in word_freq.items() if freq >= MIN_FREQ]

# Build word2idx — reserve 0 for PAD, 1 for UNK
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word in sorted(vocab_words):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(word2idx)

print(f"Total unique words in train  : {len(word_freq):,}")
print(f"Words with freq >= {MIN_FREQ}       : {len(vocab_words):,}")
print(f"Vocabulary size (with PAD+UNK): {VOCAB_SIZE:,}")
print(f"PAD index : {word2idx[PAD_TOKEN]}")
print(f"UNK index : {word2idx[UNK_TOKEN]}")

Total unique words in train  : 14,060
Words with freq >= 2       : 6,304
Vocabulary size (with PAD+UNK): 6,306
PAD index : 0
UNK index : 1


In [8]:
#Text to Sequence 
MAX_SEQUENCE_LENGTH = 32  # confirmed from EDA — covers 100% of data

def text_to_sequence(text, word2idx, max_len):
    tokens = text.split()
    # Map tokens 
    sequence = [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]
    # Truncate if longer 
    sequence = sequence[:max_len]
    # Pad if shorter 
    sequence = sequence + [word2idx[PAD_TOKEN]] * (max_len - len(sequence))
    return sequence

#Test
sample_text = train['cleaned_text'].iloc[0]
sample_seq  = text_to_sequence(sample_text, word2idx, MAX_SEQUENCE_LENGTH)

print(f"Sample cleaned text : {sample_text}")
print(f"Sequence length     : {len(sample_seq)}")
print(f"Sequence            : {sample_seq}")
print()

# Verify padding
short_text = "stock down"
short_seq  = text_to_sequence(short_text, word2idx, MAX_SEQUENCE_LENGTH)
print(f"Short text          : '{short_text}'")
print(f"Sequence length     : {len(short_seq)}")
print(f"Padded sequence     : {short_seq}")

Sample cleaned text : bynd jpmorgan reel expectation beyond meat
Sequence length     : 32
Sequence            : [847, 2974, 4514, 1967, 610, 3410, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Short text          : 'stock down'
Sequence length     : 32
Padded sequence     : [5326, 1659, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [9]:
#Convert Full Dataset to Tensors 
def dataset_to_tensors(df, word2idx, max_len):
    sequences = [text_to_sequence(text, word2idx, max_len) for text in df['cleaned_text']]
    X = torch.tensor(sequences, dtype=torch.long)
    y = torch.tensor(df['label'].values, dtype=torch.long)
    return X, y

X_train, y_train = dataset_to_tensors(train, word2idx, MAX_SEQUENCE_LENGTH)
X_valid, y_valid = dataset_to_tensors(valid, word2idx, MAX_SEQUENCE_LENGTH)

print(f"X_train shape : {X_train.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"X_valid shape : {X_valid.shape}")
print(f"y_valid shape : {y_valid.shape}")
print()
print(f"X_train dtype : {X_train.dtype}")
print(f"y_train dtype : {y_train.dtype}")
print()
print(f"y_train class counts : {torch.bincount(y_train)}")
print(f"y_valid class counts : {torch.bincount(y_valid)}")

X_train shape : torch.Size([9535, 32])
y_train shape : torch.Size([9535])
X_valid shape : torch.Size([2388, 32])
y_valid shape : torch.Size([2388])

X_train dtype : torch.int64
y_train dtype : torch.int64

y_train class counts : tensor([1442, 1923, 6170])
y_valid class counts : tensor([ 347,  475, 1566])


In [10]:
# Compute Class Weights 
class_counts = torch.bincount(y_train).float()
total        = class_counts.sum()
class_weights = total / (len(class_counts) * class_counts)

print("Class Weights for CrossEntropyLoss:")
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
for i, w in enumerate(class_weights):
    print(f"  {label_map[i]:<10} (count={int(class_counts[i]):>5}) → weight = {w:.4f}")

print()
print(f"class_weights tensor : {class_weights}")

Class Weights for CrossEntropyLoss:
  Bearish    (count= 1442) → weight = 2.2041
  Bullish    (count= 1923) → weight = 1.6528
  Neutral    (count= 6170) → weight = 0.5151

class_weights tensor : tensor([2.2041, 1.6528, 0.5151])


In [11]:
# PyTorch Dataset & DataLoader
class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_dataset = SentimentDataset(X_train, y_train)
valid_dataset = SentimentDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch size          : {BATCH_SIZE}")
print(f"Train batches       : {len(train_loader)}")
print(f"Valid batches       : {len(valid_loader)}")
print()

# Verify one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Sample batch X shape : {X_batch.shape}")
print(f"Sample batch y shape : {y_batch.shape}")
print(f"Sample batch y values: {y_batch[:10]}")

Batch size          : 64
Train batches       : 149
Valid batches       : 38

Sample batch X shape : torch.Size([64, 32])
Sample batch y shape : torch.Size([64])
Sample batch y values: tensor([2, 2, 2, 0, 0, 2, 2, 1, 1, 2])


In [12]:
# Save Preprocessing Artifacts

import os
os.makedirs('../models', exist_ok=True)

# Save vocabulary
with open('../models/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

with open('../models/idx2word.pkl', 'wb') as f:
    pickle.dump(idx2word, f)

# Save class weights
torch.save(class_weights, '../models/class_weights.pt')

# Save processed dataframes
train.to_csv('../data/train_cleaned.csv', index=False)
valid.to_csv('../data/valid_cleaned.csv', index=False)

# Save tensors
torch.save(X_train, '../models/X_train.pt')
torch.save(y_train, '../models/y_train.pt')
torch.save(X_valid, '../models/X_valid.pt')
torch.save(y_valid, '../models/y_valid.pt')

print("Saved artifacts:")
print("  ../models/word2idx.pkl")
print("  ../models/idx2word.pkl")
print("  ../models/class_weights.pt")
print("  ../data/train_cleaned.csv")
print("  ../data/valid_cleaned.csv")
print("  ../models/X_train.pt")
print("  ../models/y_train.pt")
print("  ../models/X_valid.pt")
print("  ../models/y_valid.pt")
print()

# Save config as reference for Phase 3
config = {
    'VOCAB_SIZE'          : VOCAB_SIZE,
    'MAX_SEQUENCE_LENGTH' : MAX_SEQUENCE_LENGTH,
    'BATCH_SIZE'          : BATCH_SIZE,
    'NUM_CLASSES'         : 3,
    'PAD_IDX'             : word2idx[PAD_TOKEN],
    'UNK_IDX'             : word2idx[UNK_TOKEN],
}

import json
with open('../models/config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("Config saved to ../models/config.json")
print()
print(f"Config : {json.dumps(config, indent=4)}")

Saved artifacts:
  ../models/word2idx.pkl
  ../models/idx2word.pkl
  ../models/class_weights.pt
  ../data/train_cleaned.csv
  ../data/valid_cleaned.csv
  ../models/X_train.pt
  ../models/y_train.pt
  ../models/X_valid.pt
  ../models/y_valid.pt

Config saved to ../models/config.json

Config : {
    "VOCAB_SIZE": 6306,
    "MAX_SEQUENCE_LENGTH": 32,
    "BATCH_SIZE": 64,
    "NUM_CLASSES": 3,
    "PAD_IDX": 0,
    "UNK_IDX": 1
}


In [13]:
# ─── Preprocessing Summary ───────────────────────────────────────────────────
print("=" * 60)
print("   PREPROCESSING SUMMARY")
print("=" * 60)
print(f"""
CLEANING STEPS APPLIED
  1. Lowercase
  2. Remove URLs
  3. Remove @mentions
  4. Normalize $TICKERS (remove $ symbol, keep word)
  5. Remove # symbol, keep hashtag word
  6. Remove special characters and punctuation
  7. Remove extra whitespace
  8. Tokenize
  9. Remove stopwords (kept: up/down/not/no/over/under)
  10. Lemmatize

EMPTY TEXT HANDLING
  Dropped 8 empty train rows (emojis, URL-only, mention-only)
  Final train size : 9,535
  Final valid size : 2,388

VOCABULARY
  Total unique words : 14,060
  Min frequency filter (>=2) : 6,304 words
  Vocabulary size (PAD+UNK)  : 6,306
  PAD index : 0
  UNK index : 1

SEQUENCE PREPARATION
  MAX_SEQUENCE_LENGTH : 32
  Padding             : post-padding with PAD index (0)
  Truncation          : sequences longer than 32 truncated

CLASS WEIGHTS (for CrossEntropyLoss)
  Bearish  : 2.2041
  Bullish  : 1.6528
  Neutral  : 0.5151

ARTIFACTS SAVED
  word2idx.pkl, idx2word.pkl
  class_weights.pt
  X_train.pt, y_train.pt, X_valid.pt, y_valid.pt
  train_cleaned.csv, valid_cleaned.csv
  config.json
""")


   PREPROCESSING SUMMARY

CLEANING STEPS APPLIED
  1. Lowercase
  2. Remove URLs
  3. Remove @mentions
  4. Normalize $TICKERS (remove $ symbol, keep word)
  5. Remove # symbol, keep hashtag word
  6. Remove special characters and punctuation
  7. Remove extra whitespace
  8. Tokenize
  9. Remove stopwords (kept: up/down/not/no/over/under)
  10. Lemmatize

EMPTY TEXT HANDLING
  Dropped 8 empty train rows (emojis, URL-only, mention-only)
  Final train size : 9,535
  Final valid size : 2,388

VOCABULARY
  Total unique words : 14,060
  Min frequency filter (>=2) : 6,304 words
  Vocabulary size (PAD+UNK)  : 6,306
  PAD index : 0
  UNK index : 1

SEQUENCE PREPARATION
  MAX_SEQUENCE_LENGTH : 32
  Padding             : post-padding with PAD index (0)
  Truncation          : sequences longer than 32 truncated

CLASS WEIGHTS (for CrossEntropyLoss)
  Bearish  : 2.2041
  Bullish  : 1.6528
  Neutral  : 0.5151

ARTIFACTS SAVED
  word2idx.pkl, idx2word.pkl
  class_weights.pt
  X_train.pt, y_train.pt